In [1]:
import os
import re
import json
import pandas as pd

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
PROJECT_FOLDER="/content/drive/MyDrive/CBR-PHI"

TEXT_FOLDER=os.path.join(
    PROJECT_FOLDER,
    "data",
    "processed",
    "cases_text"
)

OUTPUT_FOLDER=os.path.join(
    PROJECT_FOLDER,
    "data",
    "processed"
)

print(TEXT_FOLDER)

/content/drive/MyDrive/CBR-PHI/data/processed/cases_text


In [10]:
files=sorted(os.listdir(TEXT_FOLDER))

print("Jumlah TXT :",len(files))

Jumlah TXT : 30


In [11]:
def extract_metadata(text):

    result = {
        "nomor_perkara": "",
        "tahun": "",
        "jenis_perkara": "Perselisihan Hubungan Industrial",
        "pemohon": "",
        "termohon": "",
        "amar_putusan": ""
    }

    # Nomor Perkara
    nomor = re.search(r"nomor\s+([^\n]+)", text, re.IGNORECASE)
    if nomor:
        result["nomor_perkara"] = nomor.group(1).strip()

    # Tahun
    tahun = re.search(r"(20\d{2})", text)
    if tahun:
        result["tahun"] = tahun.group(1)

    # Pemohon
    pemohon = re.search(r"antara:(.*?)(?=melawan|lawan)", text,
                        re.IGNORECASE | re.DOTALL)

    if pemohon:
        result["pemohon"] = " ".join(
            pemohon.group(1).split()
        )[:300]

    # Termohon
    termohon = re.search(r"(melawan|lawan)(.*?)(?=mahkamah|menimbang)",
                         text,
                         re.IGNORECASE | re.DOTALL)

    if termohon:
        result["termohon"] = " ".join(
            termohon.group(2).split()
        )[:300]

    # Amar Putusan
    if "menolak permohonan kasasi" in text:
        result["amar_putusan"] = "Ditolak"

    elif "mengabulkan permohonan kasasi" in text:
        result["amar_putusan"] = "Dikabulkan"

    elif "mengabulkan sebagian" in text:
        result["amar_putusan"] = "Dikabulkan Sebagian"

    elif "tidak dapat diterima" in text:
        result["amar_putusan"] = "Tidak Diterima"

    else:
        result["amar_putusan"] = "Lainnya"

    return result

In [12]:
cases = []

for i, file in enumerate(files, start=1):

    path = os.path.join(TEXT_FOLDER, file)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    meta = extract_metadata(text)

    cases.append({

        "case_id": i,

        "nomor_perkara": meta["nomor_perkara"],

        "tahun": meta["tahun"],

        "jenis_perkara": meta["jenis_perkara"],

        "pemohon": meta["pemohon"],

        "termohon": meta["termohon"],

        "amar_putusan": meta["amar_putusan"],

        "isi_putusan": text

    })

print("Jumlah Case :", len(cases))

Jumlah Case : 30


In [13]:
df = pd.DataFrame(cases)

df.head()

,case_id,nomor_perkara,tahun,jenis_perkara,pemohon,termohon,amar_putusan,isi_putusan
0,1,k pdt sus phi g e n demi nkeadilan berdasarkan...,2022,Perselisihan Hubungan Industrial,,an kasasi dan peninjauan kembali e menghukum t...,Ditolak,a i s e n o d n i k i l b u p e a r i s g e n ...
1,2,k pdt sus phi e n n demi keadilan berdasarkan ...,2021,Perselisihan Hubungan Industrial,,l m b lisa marini warga negara indonesia dahul...,Ditolak,a i s e n o d n i k i l b u p e a r i s g e n ...
2,3,k pdt sus phi e n n demi keadilan berdasarkan ...,2021,Perselisihan Hubungan Industrial,,ane banding ataupun kasasi a r a i s m ghalama...,Ditolak,a i s e n o d n i k i l b u p e a r i s g e n ...
3,4,k pdt sus phi g e n demi nkeadilan berdasarkan...,,Perselisihan Hubungan Industrial,,dengan saksama a diajukan dalam tenggang rwakt...,Ditolak,a i s e n o d n i k i l b u p e a r i s g e n ...
4,5,k pdt sus phi g e n n demi keadilan berdasarka...,2022,Perselisihan Hubungan Industrial,,an atas putusan dalam p k perkara ini uitvoerb...,Dikabulkan,a i s e n o d n i k i l b u p e a r i s g e n ...


In [14]:
csv_path = os.path.join(OUTPUT_FOLDER, "cases.csv")

df.to_csv(csv_path,
          index=False,
          encoding="utf-8-sig")

print(csv_path)

/content/drive/MyDrive/CBR-PHI/data/processed/cases.csv


In [15]:
json_path = os.path.join(OUTPUT_FOLDER, "cases.json")

df.to_json(
    json_path,
    orient="records",
    force_ascii=False,
    indent=4
)

print(json_path)

/content/drive/MyDrive/CBR-PHI/data/processed/cases.json


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   case_id        30 non-null     int64 
 1   nomor_perkara  30 non-null     object
 2   tahun          30 non-null     object
 3   jenis_perkara  30 non-null     object
 4   pemohon        30 non-null     object
 5   termohon       30 non-null     object
 6   amar_putusan   30 non-null     object
 7   isi_putusan    30 non-null     object
dtypes: int64(1), object(7)
memory usage: 2.0+ KB


In [17]:
df.sample(5)

,case_id,nomor_perkara,tahun,jenis_perkara,pemohon,termohon,amar_putusan,isi_putusan
3,4,k pdt sus phi g e n demi nkeadilan berdasarkan...,,Perselisihan Hubungan Industrial,,dengan saksama a diajukan dalam tenggang rwakt...,Ditolak,a i s e n o d n i k i l b u p e a r i s g e n ...
15,16,k pdt sus phi g e n n demi keadilan berdasarka...,,Perselisihan Hubungan Industrial,,dengan saksama e diajukan dalam tenggang waktu...,Lainnya,a i s e n o d n i k i l b u p e a r i s g e n ...
5,6,k pdt sus phi g e demi keadilan berdasarkan ke...,2014,Perselisihan Hubungan Industrial,,dengann saksama a diajukan dalam tenggang wakt...,Ditolak,a i s e n o d n i k i l b u p e a r i s g e n ...
19,20,k pdt sus phi g e n n demi keadilan berdasarka...,2023,Perselisihan Hubungan Industrial,,an ataupun kasasi uitvoerbaar bij m b voorraad...,Lainnya,a i s e n o d n i k i l b u p e a r i s g e n ...
12,13,k pdt sus phi g e n n demi keadilan berdasarka...,,Perselisihan Hubungan Industrial,,dengan isaksama diajukan hdalam tenggang waktu...,Ditolak,a i s e n o d n i k i l b u p e a r i s g e n ...
